# AI Agents Fundamentals

**Module 11: AI Agents and MCP**  
**Objective**: Build autonomous AI agents

AI Agents:
- ReAct framework (Reasoning + Acting)
- Tool use and function calling
- Planning and task decomposition
- Memory systems
- Multi-agent architectures

## What You'll Learn
1. Agent architecture patterns
2. ReAct framework implementation
3. Tool/function calling
4. Planning algorithms
5. Agent memory systems
6. Multi-agent coordination

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Callable, Optional
import re
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. What are AI Agents?

**Agent**: Autonomous entity that perceives environment, reasons, and takes actions to achieve goals.

**Key components**:
- **Perception**: Observe environment/inputs
- **Reasoning**: Decide what to do
- **Action**: Execute decisions (use tools)
- **Memory**: Remember past interactions

**Agent vs LLM**:
- LLM: One-shot text→text
- Agent: Multi-step, can use tools, make plans, remember context

In [ ]:
def visualize_agent_architecture():
    """Visualize agent components."""
    
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('off')
    
    # Components
    components = [
        ("User Input", (0.5, 0.95), 'lightblue', 'top'),
        ("Perception", (0.5, 0.82), 'lightgreen', 'middle'),
        ("Memory", (0.15, 0.65), 'lightyellow', 'side'),
        ("Reasoning\n(LLM Brain)", (0.5, 0.65), 'plum', 'middle'),
        ("Planning", (0.85, 0.65), 'lightcoral', 'side'),
        ("Action\n(Tool Use)", (0.5, 0.48), 'orange', 'middle'),
        ("Tools", (0.2, 0.30), 'lightgray', 'tool'),
        ("Tools", (0.5, 0.30), 'lightgray', 'tool'),
        ("Tools", (0.8, 0.30), 'lightgray', 'tool'),
        ("Environment", (0.5, 0.13), 'lightcyan', 'bottom'),
    ]
    
    # Draw components
    for label, (x, y), color, comp_type in components:
        if comp_type == 'tool':
            rect = plt.Rectangle((x-0.08, y-0.04), 0.16, 0.08, 
                                color=color, ec='black', linewidth=2)
            ax.add_patch(rect)
            tool_name = ["Calculator", "Search", "Database"][int((x-0.2)/0.3)]
            ax.text(x, y, tool_name, ha='center', va='center', fontsize=9, weight='bold')
        else:
            size = 0.10 if comp_type == 'middle' else 0.08
            circle = plt.Circle((x, y), size, color=color, ec='black', linewidth=2)
            ax.add_patch(circle)
            ax.text(x, y, label, ha='center', va='center', fontsize=10, weight='bold')
    
    # Arrows
    arrows = [
        ((0.5, 0.88), (0.5, 0.75), 'Input'),
        ((0.5, 0.75), (0.5, 0.58), 'Decision'),
        ((0.5, 0.58), (0.2, 0.38), ''),
        ((0.5, 0.58), (0.5, 0.38), ''),
        ((0.5, 0.58), (0.8, 0.38), ''),
        ((0.2, 0.22), (0.5, 0.18), 'Result'),
        ((0.5, 0.22), (0.5, 0.18), 'Result'),
        ((0.8, 0.22), (0.5, 0.18), 'Result'),
        ((0.25, 0.65), (0.40, 0.65), 'Recall'),
        ((0.75, 0.65), (0.60, 0.65), 'Plan'),
    ]
    
    for start, end, label_text in arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'))
        if label_text:
            mid_x, mid_y = (start[0] + end[0])/2, (start[1] + end[1])/2
            ax.text(mid_x + 0.02, mid_y, label_text, fontsize=8, style='italic')
    
    # Feedback loop
    ax.annotate('', xy=(0.52, 0.95), xytext=(0.52, 0.20),
               arrowprops=dict(arrowstyle='->', lw=1.5, color='red', 
                             linestyle='dashed', connectionstyle="arc3,rad=.5"))
    ax.text(0.72, 0.55, 'Feedback Loop', fontsize=10, color='red', 
           rotation=-30, style='italic', weight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('AI Agent Architecture', fontsize=16, weight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    print("\nAgent Flow:")
    print("1. Perceive: User asks 'What's 15% of $453?'")
    print("2. Reason: Need calculator tool")
    print("3. Plan: Calculate 453 * 0.15")
    print("4. Act: Call calculator(453 * 0.15)")
    print("5. Observe: Result = 67.95")
    print("6. Respond: '15% of $453 is $67.95'")
    print("\n✓ Agent autonomously decided to use calculator!")

visualize_agent_architecture()

## 2. ReAct Framework

**ReAct** = Reasoning + Acting

Key idea: Interleave thoughts (reasoning) and actions in a loop.

**Pattern**:
```
Thought: I need to find X
Action: search("X")
Observation: [search results]
Thought: Now I need Y  
Action: calculate(Y)
Observation: [calculation result]
Thought: I have enough information
Answer: [final answer]
```

In [ ]:
class ReActAgent:
    """
    ReAct agent: Reasoning + Acting framework.
    
    Interleaves thoughts and actions to solve tasks.
    """
    
    def __init__(self, tools: Dict[str, Callable], llm: Optional[Callable] = None, 
                 max_iterations: int = 5):
        """
        Args:
            tools: Dictionary mapping tool names to functions
            llm: Language model for reasoning (simulated if None)
            max_iterations: Maximum reasoning steps
        """
        self.tools = tools
        self.llm = llm or self._simulated_llm
        self.max_iterations = max_iterations
        self.trace = []  # Store thought-action-observation sequence
        
    def _simulated_llm(self, prompt: str) -> str:
        """Simulated LLM responses for demonstration."""
        if "What is the capital" in prompt:
            return "Thought: I need to search for the capital.\\nAction: search('capital of France')"
        elif "capital of France" in prompt:
            return "Thought: Based on the search result, I have the answer.\\nAnswer: The capital of France is Paris."
        elif "calculate" in prompt.lower():
            return "Thought: I need to use the calculator.\\nAction: calculate('15 * 23')"
        else:
            return "Thought: I don't know how to proceed.\\nAnswer: I need more information."
    
    def _parse_action(self, text: str) -> Optional[tuple]:
        """Extract action and input from LLM output."""
        # Look for "Action: tool_name('input')"
        match = re.search(r"Action:\\s*(\\w+)\\(['\"](.+?)['\"]\\)", text)
        if match:
            tool_name, tool_input = match.groups()
            return (tool_name, tool_input)
        return None
    
    def _parse_answer(self, text: str) -> Optional[str]:
        """Extract final answer from LLM output."""
        match = re.search(r"Answer:\\s*(.+)", text)
        if match:
            return match.group(1).strip()
        return None
    
    def run(self, query: str) -> str:
        """
        Run agent on query.
        
        Args:
            query: User question
            
        Returns:
            Final answer
        """
        self.trace = []
        prompt = f"Question: {query}\\n\\n"
        
        for step in range(self.max_iterations):
            # Generate thought/action
            response = self.llm(prompt)
            self.trace.append(("thought", response))
            
            # Check for final answer
            answer = self._parse_answer(response)
            if answer:
                return answer
            
            # Parse and execute action
            action = self._parse_action(response)
            if action:
                tool_name, tool_input = action
                
                if tool_name not in self.tools:
                    observation = f"Error: Tool '{tool_name}' not found."
                else:
                    try:
                        result = self.tools[tool_name](tool_input)
                        observation = f"Observation: {result}"
                    except Exception as e:
                        observation = f"Error: {str(e)}"
                
                self.trace.append(("action", f"{tool_name}('{tool_input}')"))
                self.trace.append(("observation", observation))
                
                # Add to prompt for next iteration
                prompt += f"{response}\\n{observation}\\n\\n"
            else:
                # No action found, stop
                return "Could not determine action to take."
        
        return "Maximum iterations reached without answer."
    
    def print_trace(self):
        """Print the thought-action-observation trace."""
        print("\\nAgent Trace:")
        print("="*70)
        for i, (trace_type, content) in enumerate(self.trace, 1):
            print(f"\\nStep {i} [{trace_type.upper()}]:")
            print(content)
        print("="*70)

# Define tools
def search_tool(query: str) -> str:
    """Simulated search tool."""
    knowledge_base = {
        "capital of France": "Paris is the capital and largest city of France.",
        "capital of Japan": "Tokyo is the capital of Japan.",
        "who invented Python": "Guido van Rossum created Python in 1991."
    }
    return knowledge_base.get(query.lower(), "No results found.")

def calculator_tool(expression: str) -> str:
    """Simple calculator."""
    try:
        # Safe eval (in production, use proper math parser)
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"

# Create agent
tools = {
    "search": search_tool,
    "calculate": calculator_tool
}

agent = ReActAgent(tools)

# Test
query = "What is the capital of France?"
answer = agent.run(query)

print(f"Query: {query}")
print(f"Answer: {answer}")
agent.print_trace()

## 3. Tool/Function Calling

Modern LLMs support structured function calling.

**Pattern**:
1. Define tools with JSON schema
2. LLM outputs structured function call
3. Execute function
4. Feed result back to LLM

In [ ]:
class FunctionCallingAgent:
    """
    Agent with structured function calling.
    
    LLM outputs JSON function calls instead of text parsing.
    """
    
    def __init__(self, tools: List[Dict]):
        """
        Args:
            tools: List of tool definitions with JSON schema
        """
        self.tools = {tool['name']: tool for tool in tools}
        self.tool_functions = {}
    
    def register_function(self, name: str, func: Callable):
        """Register executable function."""
        self.tool_functions[name] = func
    
    def get_tools_schema(self) -> List[Dict]:
        """Get tools in OpenAI function calling format."""
        return [
            {
                "name": tool['name'],
                "description": tool['description'],
                "parameters": tool['parameters']
            }
            for tool in self.tools.values()
        ]
    
    def execute_function(self, function_call: Dict) -> str:
        """Execute a function call."""
        func_name = function_call['name']
        func_args = json.loads(function_call.get('arguments', '{}'))
        
        if func_name not in self.tool_functions:
            return f"Error: Function '{func_name}' not found."
        
        try:
            result = self.tool_functions[func_name](**func_args)
            return json.dumps(result)
        except Exception as e:
            return f"Error executing {func_name}: {str(e)}"

# Define tools with JSON schema (OpenAI format)
tools_schema = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g., San Francisco"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit"
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "calculate",
        "description": "Perform mathematical calculations",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Mathematical expression to evaluate"
                }
            },
            "required": ["expression"]
        }
    }
]

# Create agent
agent = FunctionCallingAgent(tools_schema)

# Register implementations
def get_weather(location: str, unit: str = "celsius") -> Dict:
    """Simulated weather API."""
    weather_data = {
        "San Francisco": {"temp": 18, "condition": "Sunny"},
        "Tokyo": {"temp": 22, "condition": "Cloudy"},
        "Paris": {"temp": 15, "condition": "Rainy"}
    }
    
    data = weather_data.get(location, {"temp": 20, "condition": "Unknown"})
    
    if unit == "fahrenheit":
        data['temp'] = data['temp'] * 9/5 + 32
    
    return {
        "location": location,
        "temperature": data['temp'],
        "unit": unit,
        "condition": data['condition']
    }

def calculate(expression: str) -> Dict:
    """Calculator function."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"result": result, "expression": expression}
    except Exception as e:
        return {"error": str(e)}

agent.register_function("get_weather", get_weather)
agent.register_function("calculate", calculate)

# Simulate function call
function_call = {
    "name": "get_weather",
    "arguments": '{"location": "San Francisco", "unit": "fahrenheit"}'
}

result = agent.execute_function(function_call)
print("Function call:")
print(json.dumps(function_call, indent=2))
print("\\nResult:")
print(result)

# Display tools schema
print("\\n\\nAvailable Tools:")
print("="*70)
for tool in agent.get_tools_schema():
    print(f"\\n{tool['name'].upper()}")
    print(f"  Description: {tool['description']}")
    print(f"  Parameters: {json.dumps(tool['parameters'], indent=4)}")

## 4. Planning Algorithms

Agents need to break down complex tasks into steps.

**Planning approaches**:
- Chain-of-Thought (sequential)
- Tree-of-Thoughts (explore multiple paths)
- Plan-and-Execute (upfront planning)

In [ ]:
class PlanningAgent:
    """
    Agent with planning capability.
    
    Decomposes complex tasks into subtasks.
    """
    
    def __init__(self, tools: Dict[str, Callable]):
        self.tools = tools
        self.plan = []
        self.execution_trace = []
    
    def create_plan(self, task: str) -> List[Dict]:
        """
        Create plan for task (simulated).
        
        In production, use LLM to generate plan.
        """
        # Simulated planning
        if "research" in task.lower():
            plan = [
                {"step": 1, "action": "search", "args": {"query": "AI history"}},
                {"step": 2, "action": "search", "args": {"query": "AI applications"}},
                {"step": 3, "action": "summarize", "args": {"topic": "AI overview"}}
            ]
        elif "calculate" in task.lower():
            plan = [
                {"step": 1, "action": "calculate", "args": {"expression": "100 * 0.15"}},
                {"step": 2, "action": "format", "args": {"value": "result"}}
            ]
        else:
            plan = [{"step": 1, "action": "search", "args": {"query": task}}]
        
        self.plan = plan
        return plan
    
    def execute_plan(self) -> List[str]:
        """Execute plan step by step."""
        results = []
        
        for step in self.plan:
            action = step['action']
            args = step['args']
            
            print(f"\\nStep {step['step']}: {action}({args})")
            
            if action in self.tools:
                result = self.tools[action](**args)
                results.append(result)
                self.execution_trace.append({
                    "step": step['step'],
                    "action": action,
                    "result": result
                })
                print(f"  Result: {result}")
            else:
                results.append(f"Tool '{action}' not available")
        
        return results
    
    def run(self, task: str) -> str:
        """Plan and execute task."""
        print(f"Task: {task}\\n")
        
        # Create plan
        plan = self.create_plan(task)
        print("Plan:")
        for step in plan:
            print(f"  {step['step']}. {step['action']}({step['args']})")
        
        # Execute
        print("\\nExecution:")
        results = self.execute_plan()
        
        # Synthesize answer (simulated)
        answer = f"Completed {len(plan)} steps. Final result: {results[-1] if results else 'None'}"
        return answer

# Define tools
def search(query: str) -> str:
    return f"Search results for '{query}'"

def summarize(topic: str) -> str:
    return f"Summary of {topic}"

def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except:
        return "Error"

def format_value(value: str) -> str:
    return f"Formatted: {value}"

tools = {
    "search": search,
    "summarize": summarize,
    "calculate": calculate,
    "format": format_value
}

# Create planning agent
planning_agent = PlanningAgent(tools)

# Run
task = "Research AI and summarize findings"
answer = planning_agent.run(task)

print(f"\\n\\nFinal Answer: {answer}")

## 5. Agent Memory

Agents need memory for context and learning.

**Memory types**:
- **Short-term**: Current conversation
- **Long-term**: Past conversations, facts
- **Episodic**: Specific experiences
- **Semantic**: General knowledge

In [ ]:
class AgentMemory:
    """
    Multi-tier memory system for agents.
    """
    
    def __init__(self, short_term_size: int = 10):
        self.short_term_size = short_term_size
        self.short_term = []  # Recent messages
        self.long_term = {}  # Key-value store
        self.episodic = []  # Past experiences
        self.semantic = {}  # Facts and knowledge
    
    def add_to_short_term(self, message: Dict):
        """Add to short-term memory (conversation buffer)."""
        self.short_term.append(message)
        if len(self.short_term) > self.short_term_size:
            # Move oldest to long-term
            oldest = self.short_term.pop(0)
            self.add_to_episodic(oldest)
    
    def add_to_long_term(self, key: str, value: str):
        """Add to long-term memory (persistent facts)."""
        self.long_term[key] = value
    
    def add_to_episodic(self, experience: Dict):
        """Add experience to episodic memory."""
        self.episodic.append(experience)
    
    def add_to_semantic(self, concept: str, definition: str):
        """Add to semantic memory (knowledge base)."""
        self.semantic[concept] = definition
    
    def recall_short_term(self) -> List[Dict]:
        """Get recent conversation."""
        return self.short_term.copy()
    
    def recall_long_term(self, key: str) -> Optional[str]:
        """Retrieve from long-term memory."""
        return self.long_term.get(key)
    
    def recall_similar_episodes(self, query: str, top_k: int = 3) -> List[Dict]:
        """Find similar past experiences (simplified)."""
        # In production, use embedding similarity
        relevant = []
        for episode in self.episodic:
            if query.lower() in str(episode).lower():
                relevant.append(episode)
        return relevant[:top_k]
    
    def recall_semantic(self, concept: str) -> Optional[str]:
        """Retrieve knowledge."""
        return self.semantic.get(concept.lower())
    
    def summarize(self) -> Dict:
        """Get memory statistics."""
        return {
            "short_term_messages": len(self.short_term),
            "long_term_facts": len(self.long_term),
            "episodic_memories": len(self.episodic),
            "semantic_concepts": len(self.semantic)
        }

# Create memory system
memory = AgentMemory(short_term_size=5)

# Simulate conversation
memory.add_to_short_term({"role": "user", "content": "What's the weather?"})
memory.add_to_short_term({"role": "assistant", "content": "It's sunny, 72°F."})
memory.add_to_short_term({"role": "user", "content": "Good for a walk?"})
memory.add_to_short_term({"role": "assistant", "content": "Yes, perfect weather!"})

# Add facts
memory.add_to_long_term("user_location", "San Francisco")
memory.add_to_long_term("user_preference", "Prefers sunny weather")

# Add knowledge
memory.add_to_semantic("temperature", "Measure of thermal energy")
memory.add_to_semantic("weather", "Atmospheric conditions")

# Add experiences (past tasks completed)
memory.add_to_episodic({
    "task": "weather_inquiry",
    "location": "San Francisco",
    "outcome": "success",
    "timestamp": "2024-01-15T10:30:00"
})

print("Memory System Status:")
print(json.dumps(memory.summarize(), indent=2))

print("\\nShort-term memory (recent conversation):")
for msg in memory.recall_short_term():
    print(f"  {msg['role']}: {msg['content']}")

print("\\nLong-term facts:")
print(f"  User location: {memory.recall_long_term('user_location')}")
print(f"  User preference: {memory.recall_long_term('user_preference')}")

print("\\nSemantic knowledge:")
print(f"  Weather: {memory.recall_semantic('weather')}")

print("\\nSimilar past experiences:")
for episode in memory.recall_similar_episodes("weather"):
    print(f"  {episode}")

## 6. Multi-Agent Systems

Multiple agents can collaborate on complex tasks.

**Patterns**:
- **Hierarchical**: Manager agent coordinates workers
- **Collaborative**: Agents work together as peers
- **Competitive**: Agents debate/critique

In [ ]:
class MultiAgentSystem:
    """
    Multi-agent system with different roles.
    """
    
    def __init__(self):
        self.agents = {}
        self.conversation = []
    
    def add_agent(self, name: str, role: str, system_prompt: str):
        """Register an agent."""
        self.agents[name] = {
            "role": role,
            "system_prompt": system_prompt
        }
    
    def agent_respond(self, agent_name: str, context: str) -> str:
        """Get response from agent (simulated)."""
        agent = self.agents[agent_name]
        role = agent['role']
        
        # Simulated responses based on role
        if role == "researcher":
            response = f"[Research] I found that {context}... Here are key findings."
        elif role == "critic":
            response = f"[Critique] Looking at {context}, I see potential issues..."
        elif role == "synthesizer":
            response = f"[Synthesis] Combining insights: {context}..."
        elif role == "manager":
            response = f"[Manager] Let's delegate: Researcher, investigate {context}."
        else:
            response = f"[{role}] Regarding {context}..."
        
        return response
    
    def run_hierarchical(self, task: str):
        """Hierarchical: Manager coordinates workers."""
        print("="*70)
        print("HIERARCHICAL MODE")
        print("="*70)
        
        # Manager plans
        manager_response = self.agent_respond("manager", task)
        print(f"\\nManager: {manager_response}")
        self.conversation.append({"agent": "manager", "message": manager_response})
        
        # Workers execute
        for worker in ["researcher", "critic"]:
            if worker in self.agents:
                response = self.agent_respond(worker, task)
                print(f"\\n{worker.capitalize()}: {response}")
                self.conversation.append({"agent": worker, "message": response})
        
        # Manager synthesizes
        final = self.agent_respond("synthesizer", task)
        print(f"\\nSynthesizer: {final}")
        self.conversation.append({"agent": "synthesizer", "message": final})
    
    def run_collaborative(self, task: str):
        """Collaborative: Agents discuss together."""
        print("\\n" + "="*70)
        print("COLLABORATIVE MODE")
        print("="*70)
        
        # Round-robin discussion
        for agent_name in self.agents:
            response = self.agent_respond(agent_name, task)
            print(f"\\n{agent_name.capitalize()}: {response}")
            self.conversation.append({"agent": agent_name, "message": response})
    
    def run_competitive(self, task: str):
        """Competitive: Agents debate."""
        print("\\n" + "="*70)
        print("COMPETITIVE MODE (Debate)")
        print("="*70)
        
        # Two sides debate
        pro_response = self.agent_respond("researcher", f"Argue for: {task}")
        print(f"\\nPro (Researcher): {pro_response}")
        
        con_response = self.agent_respond("critic", f"Argue against: {task}")
        print(f"\\nCon (Critic): {con_response}")
        
        # Judge decides
        judge_response = f"[Judge] Both sides present valid points. Decision: ..."
        print(f"\\nJudge: {judge_response}")

# Create multi-agent system
mas = MultiAgentSystem()

# Add agents with different roles
mas.add_agent("manager", "manager", "You coordinate tasks and delegate work.")
mas.add_agent("researcher", "researcher", "You research and gather information.")
mas.add_agent("critic", "critic", "You critically analyze and find flaws.")
mas.add_agent("synthesizer", "synthesizer", "You combine insights into final answer.")

# Run different modes
task = "Should we adopt AI in healthcare?"

mas.run_hierarchical(task)
mas.run_collaborative(task)
mas.run_competitive(task)

## Summary

You've mastered AI agent fundamentals:
- ✅ Agent architecture (Perception, Reasoning, Action, Memory)
- ✅ ReAct framework (Thought → Action → Observation loop)
- ✅ Function calling with JSON schemas
- ✅ Planning algorithms (sequential, tree-based)
- ✅ Memory systems (short-term, long-term, episodic, semantic)
- ✅ Multi-agent patterns (hierarchical, collaborative, competitive)

**Key Insights**:
1. **Agents** = LLMs + Tools + Memory + Planning
2. **ReAct** interleaves reasoning and acting
3. **Function calling** provides structured tool use
4. **Memory** enables context and learning
5. **Multi-agent** systems handle complex tasks through collaboration

**Best Practices**:
1. Use ReAct for multi-step reasoning
2. Provide clear tool descriptions
3. Limit max iterations to prevent infinite loops
4. Implement memory for context retention
5. Use hierarchical agents for complex workflows

**Agent Types by Use Case**:
- **Simple QA**: Single-agent with search tool
- **Research**: Multi-step ReAct agent
- **Code generation**: Agent with code execution sandbox
- **Customer support**: Agent with knowledge base + memory
- **Complex analysis**: Multi-agent collaboration

**Next Notebook**: Model Context Protocol (MCP) for standardized tool integration

## Exercises

1. **Implement Tree-of-Thoughts**: Add branching exploration to planning
2. **Build code agent**: Agent that writes and executes Python code
3. **Add vector memory**: Store experiences as embeddings for similarity search